# Model Development and Validation

Machine learning models were developed using molecular descriptors, Morgan fingerprints, and combined feature representations to predict anti-leishmanial activity (pIC50).

In [1]:
import pandas as pd

X_desc = pd.read_csv("../data_clean/X_descriptors.csv")
X_fp = pd.read_csv("../data_clean/X_fingerprints.csv")
X_combined = pd.read_csv("../data_clean/X_combined.csv")
y = pd.read_csv("../data_clean/y_pIC50.csv")

print(X_desc.shape)
print(X_fp.shape)
print(X_combined.shape)
print(y.shape)

(5457, 11)
(5457, 1024)
(5457, 1035)
(5457, 1)


## Train-Test Split

The dataset was divided into training and testing sets. The training set was used for model development, while the testing set was reserved for independent performance evaluation.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_desc,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(4365, 11)
(1092, 11)


## Random Forest Regression

A Random Forest Regressor was trained using molecular descriptors as input features. Random Forest was selected because it can model complex non-linear relationships between molecular properties and biological activity.

In [4]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train.values.ravel())

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [5]:
y_pred = rf.predict(X_test)

## Model Evaluation

Model performance was assessed using the coefficient of determination (R²), mean absolute error (MAE), and root mean squared error (RMSE).

In [6]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("R²:", round(r2, 3))
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))

R²: 0.471
MAE: 0.458
RMSE: 0.628


## Cross-Validation

Five-fold cross-validation was performed to assess model robustness and estimate predictive performance across different subsets of the dataset.

In [8]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf,
    X_desc,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("CV Scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())

CV Scores: [0.08804742 0.21137803 0.215264   0.27690901 0.13323028]
Mean CV R²: 0.18496574888618306


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)


X_train, X_test, y_train, y_test = train_test_split(
    X_fp,
    y,
    test_size=0.2,
    random_state=42
)


rf_fp = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

rf_fp.fit(
    X_train,
    y_train.values.ravel()
)


y_pred = rf_fp.predict(X_test)


r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)


cv_scores = cross_val_score(
    rf_fp,
    X_fp,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("R²:", round(r2, 3))
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("CV Scores:", cv_scores)
print("Mean CV R²:", round(cv_scores.mean(), 3))

R²: 0.613
MAE: 0.386
RMSE: 0.537
CV Scores: [0.27395387 0.32265079 0.33402316 0.36196426 0.21935219]
Mean CV R²: 0.302


Morgan fingerprints substantially outperformed traditional molecular descriptors in predicting anti-leishmanial activity. This suggests that structural subfeatures and local atomic environments contribute more strongly to biological activity than global physicochemical descriptors alone.

In [10]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y,
    test_size=0.2,
    random_state=42
)

# Random Forest model
rf_combined = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

# Train model
rf_combined.fit(
    X_train,
    y_train.values.ravel()
)

# Predictions
y_pred = rf_combined.predict(X_test)

# Evaluation metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"R²   : {r2:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")

# 5-fold Cross Validation
cv_scores = cross_val_score(
    rf_combined,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("\n" + "="*50)
print("5-FOLD CROSS VALIDATION")
print("="*50)
print("CV Scores:", cv_scores)
print(f"Mean CV R² : {cv_scores.mean():.3f}")
print(f"Std CV R²  : {cv_scores.std():.3f}")

TEST SET PERFORMANCE
R²   : 0.616
MAE  : 0.382
RMSE : 0.534

5-FOLD CROSS VALIDATION
CV Scores: [0.26543262 0.32744346 0.37222993 0.3678229  0.21953597]
Mean CV R² : 0.310
Std CV R²  : 0.059


In [11]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Random Forest base model
rf = RandomForestRegressor(random_state=42, n_jobs=1)

# Hyperparameter search space
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.3, 0.5, None]
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=25,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=1,
    verbose=1
)

random_search.fit(X_train, y_train.values.ravel())

print("Best Parameters:", random_search.best_params_)
print("Best CV R²:", round(random_search.best_score_, 3))

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': None}
Best CV R²: 0.571


In [12]:
# Best tuned model
best_rf = random_search.best_estimator_

# Predictions on test set
y_pred_tuned = best_rf.predict(X_test)

# Metrics
r2_tuned = r2_score(y_test, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = root_mean_squared_error(y_test, y_pred_tuned)

print("="*50)
print("TUNED RANDOM FOREST TEST PERFORMANCE")
print("="*50)
print(f"R²   : {r2_tuned:.3f}")
print(f"MAE  : {mae_tuned:.3f}")
print(f"RMSE : {rmse_tuned:.3f}")

TUNED RANDOM FOREST TEST PERFORMANCE
R²   : 0.623
MAE  : 0.379
RMSE : 0.530


In [13]:
print("="*50)
print("BASELINE vs TUNED RANDOM FOREST")
print("="*50)
print(f"Baseline Test R² : {r2:.3f}")
print(f"Tuned Test R²    : {r2_tuned:.3f}")
print(f"Baseline MAE     : {mae:.3f}")
print(f"Tuned MAE        : {mae_tuned:.3f}")
print(f"Baseline RMSE    : {rmse:.3f}")
print(f"Tuned RMSE       : {rmse_tuned:.3f}")

BASELINE vs TUNED RANDOM FOREST
Baseline Test R² : 0.616
Tuned Test R²    : 0.623
Baseline MAE     : 0.382
Tuned MAE        : 0.379
Baseline RMSE    : 0.534
Tuned RMSE       : 0.530


## Hyperparameter Optimization of Random Forest

RandomizedSearchCV was used to optimize Random Forest hyperparameters. The tuned model achieved a modest improvement over the baseline model, suggesting that feature representation had a greater impact on predictive performance than parameter optimization.

## XGBoost Regression

An Extreme Gradient Boosting (XGBoost) model was trained using the combined descriptor-fingerprint representation. XGBoost is a powerful ensemble learning algorithm capable of capturing complex non-linear relationships and feature interactions.

In [15]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y,
    test_size=0.2,
    random_state=42
)

# XGBoost model
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1
)

# Train
xgb.fit(
    X_train,
    y_train.values.ravel()
)

# Predict
y_pred_xgb = xgb.predict(X_test)

# Metrics
r2_xgb = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = root_mean_squared_error(y_test, y_pred_xgb)

print("="*50)
print("XGBOOST TEST PERFORMANCE")
print("="*50)
print(f"R²   : {r2_xgb:.3f}")
print(f"MAE  : {mae_xgb:.3f}")
print(f"RMSE : {rmse_xgb:.3f}")

XGBOOST TEST PERFORMANCE
R²   : 0.618
MAE  : 0.393
RMSE : 0.533


In [16]:
cv_scores_xgb = cross_val_score(
    xgb,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("CV Scores:", cv_scores_xgb)
print("Mean CV R²:", round(cv_scores_xgb.mean(), 3))
print("Std CV R² :", round(cv_scores_xgb.std(), 3))

CV Scores: [0.25155353 0.34019959 0.3924282  0.40765591 0.25629389]
Mean CV R²: 0.33
Std CV R² : 0.066


In [17]:
print("="*60)
print("MODEL COMPARISON")
print("="*60)

print(f"Random Forest R² : {r2_tuned:.3f}")
print(f"XGBoost R²       : {r2_xgb:.3f}")

print(f"Random Forest RMSE : {rmse_tuned:.3f}")
print(f"XGBoost RMSE       : {rmse_xgb:.3f}")

MODEL COMPARISON
Random Forest R² : 0.623
XGBoost R²       : 0.618
Random Forest RMSE : 0.530
XGBoost RMSE       : 0.533


While the tuned Random Forest model achieved the highest test-set performance, XGBoost produced the highest cross-validated R² score, suggesting greater robustness and generalization across different data partitions.

In [18]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

extra = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=1
)

extra.fit(
    X_train,
    y_train.values.ravel()
)

y_pred_extra = extra.predict(X_test)

r2_extra = r2_score(y_test, y_pred_extra)
mae_extra = mean_absolute_error(y_test, y_pred_extra)
rmse_extra = root_mean_squared_error(y_test, y_pred_extra)

cv_scores_extra = cross_val_score(
    extra,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("R²:", round(r2_extra, 3))
print("MAE:", round(mae_extra, 3))
print("RMSE:", round(rmse_extra, 3))
print("Mean CV R²:", round(cv_scores_extra.mean(), 3))
print("Std CV R²:", round(cv_scores_extra.std(), 3))

R²: 0.622
MAE: 0.368
RMSE: 0.531
Mean CV R²: 0.3
Std CV R²: 0.059


In [19]:
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

cat = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=0,
    random_state=42
)

cat.fit(
    X_train,
    y_train.values.ravel()
)

y_pred_cat = cat.predict(X_test)

r2_cat = r2_score(y_test, y_pred_cat)
mae_cat = mean_absolute_error(y_test, y_pred_cat)
rmse_cat = root_mean_squared_error(y_test, y_pred_cat)

cv_scores_cat = cross_val_score(
    cat,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("R²:", round(r2_cat, 3))
print("MAE:", round(mae_cat, 3))
print("RMSE:", round(rmse_cat, 3))
print("Mean CV R²:", round(cv_scores_cat.mean(), 3))
print("Std CV R²:", round(cv_scores_cat.std(), 3))

R²: 0.542
MAE: 0.441
RMSE: 0.584
Mean CV R²: 0.301
Std CV R²: 0.071


In [20]:
results = pd.DataFrame({
    "Model": [
        "RF Descriptors",
        "RF Fingerprints",
        "RF Combined",
        "RF Tuned",
        "XGBoost",
        "Extra Trees",
        "CatBoost"
    ],
    "Test_R2": [
        0.471,
        0.613,
        0.616,
        0.623,
        0.618,
        0.622,
        0.542
    ]
})

results

,Model,Test_R2
0,RF Descriptors,0.471
1,RF Fingerprints,0.613
2,RF Combined,0.616
3,RF Tuned,0.623
4,XGBoost,0.618
5,Extra Trees,0.622
6,CatBoost,0.542
